In [ ]:
# Figure 4 - breakdown of non-uniform twin search results by hyperedge size distribution

In [ ]:
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gs

%matplotlib inline

In [ ]:
sup_title_size = 20
label_size = 18
tick_size = 14
legend_font = 16
labelpad = 3
mpl.rcParams["figure.titlesize"] = sup_title_size
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size
plt.rcParams['lines.markersize'] = 12

In [ ]:
import sys
sys.path.append("../python/")
from distance_stats import read_data

In [ ]:
def get_proj_and_sizes(hg, n):
    proj = np.zeros((n, n))
    size_dist = [0 for i in range(n-1)]
    for he in hg:
        if max(he) > n:
            print(he)

        k = len(he)
        # Update the size distribution
        size_dist[k-2] += 1

        # Update the projection
        for i in range(k-1):
            for j in range(i+1, k):
                proj[he[i],he[j]] += 1
                proj[he[j],he[i]] += 1

        # Include the diagonal
        for i in range(k):
            proj[he[i],he[i]] += 1

    
    return proj, tuple(size_dist)

def count_k_uniform_diagonals(output_data, k=None, k_unif_example=None, norm_by_df=False):
    n = output_data["n"][0]
    # We know that the diagonal of any k-uniform hypergraph
    # is the same as the input diagonal
    k_unif_diags = dict()
    pre_k_unif = []
    # For each sample
    for proj_idx in range(len(output_data["n"])):
        hgs = output_data["hypergraphs"][proj_idx]
        # For all of the hypergraphs in the sample
        for hg in hgs:
            # Compute the projection and the size distribution
            proj, size_dist = get_proj_and_sizes(hg, n)

            # If k-uniform, save it
            if sum(size_dist) == size_dist[k-2]:
                k_unif_example = proj

            if k_unif_example is not None:
                # Otherwise, check the diagonal against 3-uniform example
                equiv_diag = True
                for i in range(n):
                    if k_unif_example[i,i] != proj[i,i]:
                        equiv_diag = False

                if equiv_diag:
                    if not size_dist in k_unif_diags:
                        k_unif_diags[size_dist] = 1
                    else:
                        k_unif_diags[size_dist] += 1
            else:
                pre_k_unif.append((proj, size_dist))


        if k_unif_example is None:
            print("Never found k-uniform example??")
            print(hgs)
        
        for proj, size_dist in pre_k_unif:
            equiv_diag = True
            for i in range(n):
                if k_unif_example[i,i] != proj[i,i]:
                    equiv_diag = False
                    break
            
            if equiv_diag:
                if not size_dist in k_unif_diags:
                    k_unif_diags[size_dist] = 1
                else:
                    k_unif_diags[size_dist] += 1
    
    if norm_by_df:
        for size_dist in k_unif_diags.keys():
            k_unif_diags[size_dist] /= len(output_data["n"])
    
    return k_unif_diags

In [ ]:
def get_size_dist_freqs(results_path, filename, limit_m_hyperedges):
    output_data = read_data(results_path + filename, xgi_hypergraphs=False)
    size_dist_frequencies = dict()
    size_dist_frequencies_unf = dict()
    # On the x-axis will be each histogram, then a bar for the frequency
    for dist_results_dict in output_data["dist_results_dict"]:
        for size_dist, info in dist_results_dict.items():
            if limit_m_hyperedges and sum([tup[1] for tup in size_dist]) != m:
                continue
    
            if not size_dist in size_dist_frequencies:
                size_dist_frequencies[size_dist] = info["filtered"]
                size_dist_frequencies_unf[size_dist] = info["unfiltered"]
            else:
                size_dist_frequencies[size_dist] += info["filtered"]
                size_dist_frequencies_unf[size_dist] += info["unfiltered"]

    return output_data, size_dist_frequencies, size_dist_frequencies_unf

In [ ]:
def get_histograms(size_dist_frequencies, max_x, limit_m_hyperedges, offset=0, col_seed=800):
    # bottom axis: Histogram per distribution
    offset = 0
    bar_colors = []
    x = []
    xticks = []
    y = []
    rng = np.random.default_rng(seed=col_seed)
    for size_dist, freq in size_dist_frequencies:
        color_idx = rng.integers(0, 20)
        for he_size, dval in sorted(size_dist):
            x.append(he_size+offset)
            y.append(dval)
            bar_colors.append(plt.cm.tab20c(color_idx))
    
        for he_size in range(2, max_x+1):
            xticks.append(he_size)
    
        xticks.append("")
        offset += max_x
    
    xticks_tup = None
    if limit_m_hyperedges:
        xticks_tup=(list(range(2, len(xticks)+2)), xticks)
        

    return x, y, bar_colors, xticks_tup

def get_twin_data_bars(size_dist_frequencies, size_dist_frequencies_unf, k_unif_diags, max_x):
    x_isos = []
    x_twins = []
    x_diags = []
    y_isos = []
    y_twins = []
    y_diags = []

    center_point = (max_x-2) / 2
    idx = 1
    for size_dist_tups, freq in size_dist_frequencies:
        x_twins.append(1 + center_point*idx)
        x_isos.append(1 + center_point*idx)
        
        y_twins.append(size_dist_frequencies_unf[size_dist_tups])
        y_isos.append(freq)
        
        size_dist = np.zeros(n-1)
        for size, count in size_dist_tups:
            size_dist[size-2] = count

        size_dist = tuple([int(v) for v in size_dist])
        if size_dist in k_unif_diags:
            x_diags.append(1 + center_point*idx)
            y_diags.append(k_unif_diags[size_dist])
    
        idx += max_x / 2
    return x_isos, x_twins, x_diags, y_isos, y_twins, y_diags

In [ ]:
k = 3
n = 6
results_path = "../../results/reproduce/exhaustive/"

m = 10
norm_by_df = True

# Get a dictionary from histogram of hyperedge sizes to frequency of each
# Read the dataframe
filename = f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections.csv"

In [ ]:
for limit_m_hyperedges in [True, False]:
    output_data, size_dist_frequencies, size_dist_frequencies_unf = get_size_dist_freqs(results_path, filename, limit_m_hyperedges)
    
    k_unif_diags = count_k_uniform_diagonals(output_data, k=k, norm_by_df=norm_by_df)
    
    # sort the distributions by number of twins
    size_dist_frequencies = sorted(size_dist_frequencies.items(), reverse=True,
        key=lambda kv: kv[1])
    
    max_x=n
    
    fig = plt.figure(figsize=(10, 4), constrained_layout=True)
    gs = fig.add_gridspec(2, 1, height_ratios=[0.6, 0.4], hspace=0.1)
    x_isos, x_twins, x_diags, y_isos, y_twins, y_diags = get_twin_data_bars(size_dist_frequencies,
                                                                            size_dist_frequencies_unf,
                                                                            k_unif_diags,
                                                                            max_x)
    
    x, y, bar_colors, xticks_tup = get_histograms(size_dist_frequencies, max_x, limit_m_hyperedges, offset=0, col_seed=800)
    
    ax_short = fig.add_subplot(gs[1, :])
    ax_short.bar(x, y,
        color=bar_colors,
        alpha=0.5,
        edgecolor="black",
    )
    
    if limit_m_hyperedges:
        ax_short.set_xticks(xticks_tup[0], xticks_tup[1])
        ax_short.tick_params(axis="x", length=2)
    else:
        xtick_pos = []
        for x_pos in map(int, x_twins):
            #xtick_pos.extend([x_pos-2, x_pos+2])
            xtick_pos.extend(list(range(x_pos-2, x_pos+2+1)))
        ax_short.set_xticks(xtick_pos, [])
        #ax_short.set_xticks(x, [])
        ax_short.tick_params(axis="x", length=1.5)
    
    ax_short.set_ylabel("Freq.")
    ax_short.spines[['top', 'right']].set_visible(False)
    
    
    ax_tall = fig.add_subplot(gs[0, :])
    ax_tall.scatter(x_twins, y_twins, marker='o', facecolor='none', edgecolor="#FF8C00", label="Twins")
    ax_tall.scatter(x_isos, y_isos, marker='+', facecolor='#91bfdb', label="Iso. Classes")
    ax_tall.scatter(x_diags, y_diags, marker="D", facecolor='none', edgecolor="#AF8DC3", label="Matching Diag")
    ax_tall.spines[['top', 'right']].set_visible(False)
    ax_tall.set_ylabel("Freq.")
    ax_tall.set_xticks([])
    #ax_tall.set_yticks([round(yt, -2) for yt in np.linspace(1, max(y_twins)*1.25, 5)])
    yt = ax_tall.get_yticks()
    yt[yt==0] = 1
    ax_tall.set_yticks(yt)
    
    ax_tall.grid(visible=True, which='major', axis='both', color='black', alpha=0.1, linestyle='-')
    
    if limit_m_hyperedges:
        text_label_x = 0.05
        text_label_y = 1.0
        plot_label = "(a)"
    else:
        text_label_x = 0.05
        text_label_y = 1.1
        plot_label = "(b)"
        
    ax_tall.text(text_label_x, text_label_y, plot_label, size=label_size, fontweight="bold", horizontalalignment='center',
         verticalalignment='center', transform=ax_tall.transAxes)
    
    ax_tall.set_xlim(-1)
    ax_short.set_xlim(-1)
    
    if limit_m_hyperedges:
        ax_short.set_xlabel("Hyperedge Size Distribution")
        ax_tall.set_ylim(-2000, max(y_twins)*1.25)
        ax_tall.legend(framealpha=1.0)
        fig.suptitle(r"Twins with exactly $m=10$ hyperedges")
        fig.savefig("../plots/k-3_n-6_m-10_non-uniform_exhaustive_projections-sum-m.pdf")
    else:
        ax_short.set_xlabel("Hyperedge Size Distribution")
        ax_tall.set_ylim(-10000, max(y_twins)*1.25)
        fig.suptitle(r"All Twins")
        fig.savefig("../plots/k-3_n-6_m-10_non-uniform_exhaustive_projections.pdf")